# Supabase client

Uses the official Python client (REST + Auth + Realtime). Set these in `.env` (see `.env.example`):

- **`SUPABASE_URL`** — Project URL from Supabase Dashboard → Settings → API (e.g. `https://<project-ref>.supabase.co`).
- **`SUPABASE_ANON_KEY`** — `anon` `public` key from the same page.

Install deps once: `pip install -r requirements.txt` (includes `supabase` and `python-dotenv`).

In [3]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from supabase import Client, create_client

# Find .env from current working directory upward.
_env = find_dotenv(filename=".env", usecwd=True)
if _env:
    load_dotenv(_env)
else:
    load_dotenv()

SUPABASE_URL = (os.environ.get("SUPABASE_URL") or "").strip().strip('"').strip("'")
SUPABASE_ANON_KEY = (os.environ.get("SUPABASE_ANON_KEY") or "").strip().strip('"').strip("'")

missing = [
    name
    for name, value in {
        "SUPABASE_URL": SUPABASE_URL,
        "SUPABASE_ANON_KEY": SUPABASE_ANON_KEY,
    }.items()
    if not value
]

if missing:
    expected_env = Path.cwd() / ".env"
    raise ValueError(
        "Missing env vars: "
        + ", ".join(missing)
        + f". Create {expected_env} (or project-root .env) from .env.example and fill real values."
    )

# Allow common input without scheme, e.g. project-ref.supabase.co
if SUPABASE_URL.startswith("db."):
    SUPABASE_URL = "https://" + SUPABASE_URL[len("db."):]
elif not SUPABASE_URL.startswith(("http://", "https://")):
    SUPABASE_URL = "https://" + SUPABASE_URL

SUPABASE_URL = SUPABASE_URL.rstrip("/")

if ".supabase.co" not in SUPABASE_URL:
    raise ValueError(
        f"SUPABASE_URL looks invalid: {SUPABASE_URL}. Expected like https://<project-ref>.supabase.co"
    )

SUPABASE_SERVICE_ROLE_KEY = (os.environ.get("SUPABASE_SERVICE_ROLE_KEY") or "").strip().strip('"').strip("'")

# Default to anon (what your frontend typically uses). Optionally create a service-role client
# for debugging if you provided SUPABASE_SERVICE_ROLE_KEY in .env.
supabase_anon: Client = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
supabase_service: Client | None = (
    create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY) if SUPABASE_SERVICE_ROLE_KEY else None
)

supabase: Client = supabase_anon
print(
    "Client OK -",
    SUPABASE_URL.split("//")[-1].split(".")[0] + "...",
    "(key=anon" + ("/service_role" if supabase_service else ")"),
)

Client OK - lllfdmraatophjxbzywm... (key=anon)


In [4]:
TABLES = ["Dim_Source", "Dim_Product", "Fact_Source_Product"]

# Read-only check using anon key
print("--- anon key ---")
for table_name in TABLES:
    try:
        resp = supabase_anon.table(table_name).select("*", count="exact").limit(1).execute()
        print(f"{table_name}: {resp.count} rows")
    except Exception as exc:
        print(f"{table_name}: read error (anon) -> {exc}")

--- anon key ---
Dim_Source: 6 rows
Dim_Product: 2764 rows
Fact_Source_Product: 2774 rows


In [5]:
# Select from Dim_Source
resp_dim_source = supabase.table("Dim_Source").select("*").limit(10).execute()
resp_dim_source.data

[{'SourceID': 'S01', 'SourceName': 'An Phat PC'},
 {'SourceID': 'S02', 'SourceName': 'CellphoneS'},
 {'SourceID': 'S03', 'SourceName': 'FPT Shop'},
 {'SourceID': 'S04', 'SourceName': 'Gear VN'},
 {'SourceID': 'S05', 'SourceName': 'Phong Vu'},
 {'SourceID': 'S06', 'SourceName': 'The Gioi Di Dong'}]

In [6]:
# Select from Dim_Product
resp_dim_product = supabase.table("Dim_Product").select("*").limit(10).execute()
resp_dim_product.data

[{'ProductID': 'P0001',
  'ProductName': 'Laptop ASUS ExpertBook P1503CVA-C5H08-50W (Intel Core 5 Processor 210H | Intel Graphics | 15.6 inch FHD | 8GB | 512GB | Win 11 | Xám)',
  'Brand': 'Asus',
  'Category': 'Office',
  'CPUBrand': 'Intel',
  'CPUFamily': 'Intel Core',
  'GPUType': 'Integrated',
  'GPUModel': 'Intel Arc / Integrated',
  'RamCapacity': '8.0',
  'RamType': 'DDR5 / LPDDR5',
  'StorageCapacity': '512.0',
  'StorageType': 'SSD',
  'DisplaySize': '15.6" - 16.0"',
  'DisplayResolution': 'FHD / FHD+',
  'DisplayRefreshRate': '60 - 90',
  'OSFamily': 'Windows',
  'DesignWeight': '1.68',
  'IsAiLaptop': 'False',
  'IsAvailable': 'True'},
 {'ProductID': 'P0002',
  'ProductName': 'Laptop HP OmniBook X Flip 14-fm0088TU BZ7Q2PA (Intel Core Ultra 5 226V | 16GB | 512GB | Intel Arc | 14 inch 2K | Cám ứng | Win 11 | Office | Xanh)',
  'Brand': 'HP',
  'Category': 'Office',
  'CPUBrand': 'Intel',
  'CPUFamily': 'Intel Core Ultra',
  'GPUType': 'Integrated',
  'GPUModel': 'Intel Arc / 

In [7]:
# Select from Fact_Source_Product
resp_fact_source_product = supabase.table("Fact_Source_Product").select("*").limit(10).execute()
resp_fact_source_product.data

[{'SourceID': 'S01',
  'ProductID': 'P0001',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 615,
  'CurrentPrice': 18690000},
 {'SourceID': 'S01',
  'ProductID': 'P0002',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 6625,
  'CurrentPrice': 27590000},
 {'SourceID': 'S01',
  'ProductID': 'P0003',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 159,
  'CurrentPrice': 35990000},
 {'SourceID': 'S01',
  'ProductID': 'P0004',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 5998,
  'CurrentPrice': 17690000},
 {'SourceID': 'S01',
  'ProductID': 'P0005',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 886,
  'CurrentPrice': 18990000},
 {'SourceID': 'S01',
  'ProductID': 'P0006',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 1192,
  'CurrentPrice': 52910000},
 {'SourceID': 'S01',
  'ProductID': 'P0007',
  'AvgRating': 0,
  'ReviewCount': 0,
  'SatisfiedCount': 9504,
  'CurrentPrice': 19990000},
 {'SourceID': 'S01',
  'ProductID': '